# 18.4 Практическая работа

## Очистка данных и выявление аномалий

Цель работы - оценить качество входной витрины страховых полисов, обработать пропущенные значения, выявить аномальные записи и сохранить очищенную версию данных.

В итоговой таблице необходимо:
- заполнить необходимые пропуски;
- выявить аномальные наблюдения;
- добавить бинарный признак `anomaly`;
- сохранить результат в Excel.

## 1. Загрузка и первичный анализ данных

Загрузим исходный Excel-файл и изучим:
- размер таблицы;
- названия столбцов;
- типы данных;
- первые строки;
- количество пропущенных значений.

In [1]:
import pandas as pd
import numpy as np

df = pd.read_excel("input_data.xlsx")

print("Размер таблицы:", df.shape)

print("\nСтолбцы:")
print(df.columns.tolist())

print("\nТипы данных:")
print(df.dtypes)

print("\nПропуски:")
print(df.isna().sum())

display(df.head())

Размер таблицы: (3711, 16)

Столбцы:
['Unnamed: 0', 'contract_num', 'product_name', 'client_id', 'contract_status', 'currency_name', 'duration', 'country', 'age', 'sex', 'price', 'insurance_amount', 'loss_payout_amt', 'price_usd', 'insurance_amount_usd', 'loss_payout_amt_usd']

Типы данных:
Unnamed: 0                int64
contract_num             object
product_name             object
client_id                 int64
contract_status          object
currency_name            object
duration                  int64
country                  object
age                       int64
sex                      object
price                     int64
insurance_amount          int64
loss_payout_amt         float64
price_usd               float64
insurance_amount_usd    float64
loss_payout_amt_usd     float64
dtype: object

Пропуски:
Unnamed: 0                 0
contract_num               0
product_name               0
client_id                  0
contract_status            0
currency_name             

,Unnamed: 0,contract_num,product_name,client_id,contract_status,currency_name,duration,country,age,sex,price,insurance_amount,loss_payout_amt,price_usd,insurance_amount_usd,loss_payout_amt_usd
0,0,ТТЕ7227715*****,Страхование путешественников,10161870404,Действует,Российский рубль,10,Беларусь,20,M,1096,2000000,NaN,12.41768,22660.000080,NaN
1,1,ТТЕ7227715*****,Страхование путешественников,1017159879,Действует,Российский рубль,7,Индонезия,55,M,1918,5000000,NaN,21.73094,56650.000199,NaN
2,2,БАДАМСТЕ55*****,Страхование путешественников,10161883357,Действует,Российский рубль,10,Беларусь,61,F,1096,2000000,NaN,12.41768,22660.000080,NaN
3,3,ТТЕ7227715*****,Страхование путешественников,100766461,Действует,Российский рубль,14,Грузия,25,M,1534,2000000,NaN,17.38022,22660.000080,NaN
4,4,ТТЕ7227715*****,Страхование путешественников,10171604758,Действует,Российский рубль,7,Аргентина,33,M,1918,5000000,NaN,21.73094,56650.000199,NaN


## 2. Анализ причин пропусков

Перед заполнением пропусков проверим их смысл.

Особое внимание уделим:
- `price_usd`;
- `insurance_amount_usd`;
- `loss_payout_amt`;
- `loss_payout_amt_usd`.

Проверим, связаны ли пропуски с валютой договора и наличием страхового убытка.

In [2]:
print("Валюты:")
print(df["currency_name"].value_counts(dropna=False))

print("\nПропуски price_usd по валютам:")
print(
    df[df["price_usd"].isna()]
    ["currency_name"]
    .value_counts(dropna=False)
)

print("\nПропуски insurance_amount_usd по валютам:")
print(
    df[df["insurance_amount_usd"].isna()]
    ["currency_name"]
    .value_counts(dropna=False)
)

print("\nСтрок с loss_payout_amt:")
print(df["loss_payout_amt"].notna().sum())

print("\nСтрок с loss_playout_amt_usd:")
print(df["loss_payout_amt_usd"].notna().sum())

Валюты:
currency_name
Российский рубль    3591
Доллар США           120
Name: count, dtype: int64

Пропуски price_usd по валютам:
currency_name
Доллар США    120
Name: count, dtype: int64

Пропуски insurance_amount_usd по валютам:
currency_name
Доллар США    120
Name: count, dtype: int64

Строк с loss_payout_amt:
45

Строк с loss_playout_amt_usd:
43


In [3]:
display(
    df.loc[
        df["loss_payout_amt"].notna()
        | df["loss_payout_amt_usd"].notna(),
        [
            "currency_name",
            "loss_payout_amt",
            "loss_payout_amt_usd"
        ]
    ].head(20)
)

,currency_name,loss_payout_amt,loss_payout_amt_usd
16,Российский рубль,100000.0,1133.000004
32,Российский рубль,50000.0,566.500002
90,Доллар США,20000.0,NaN
100,Российский рубль,30000.0,339.900001
109,Российский рубль,70000.0,793.100003
245,Российский рубль,60000.0,679.800002
289,Российский рубль,50000.0,566.500002
472,Российский рубль,100000.0,1133.000004
579,Российский рубль,50000.0,566.500002
698,Российский рубль,20000.0,226.600001


## 3. Заполнение пропущенных значений

Анализ показал, что пропуски имеют разную природу.

### `price_usd` и `insurance_amount_usd`

Все 120 пропусков относятся к договорам, валюта которых уже указана как `Доллар США`.

Следовательно, пересчёт для этих строк не требуется: долларовое значение должно совпадать с исходным.

### `loss_payout_amt`

Значение присутствует только у 45 договоров. В остальных случаях отсутствие значения означает, что страховой выплаты не было.

Поэтому такие пропуски логично заменить на `0`.

### `loss_payout_amt_usd`

Для большинства договоров отсутствие значения также означает отсутствие выплаты.

Кроме того, у двух долларовых договоров имеется исходная выплата `loss_payout_amt`, но отсутствует значение в `loss_payout_amt_usd`. Так как исходная валюта этих договоров уже доллар США, значение можно перенести без пересчёта.

In [4]:
usd_mask = df["currency_name"] == "Доллар США"

df.loc[
    usd_mask & df["price_usd"].isna(),
    "price_usd"
] = df.loc[
    usd_mask & df["price_usd"].isna(),
    "price"
]

df.loc[
    usd_mask & df["insurance_amount_usd"].isna(),
    "insurance_amount_usd"
] = df.loc[
    usd_mask & df["insurance_amount_usd"].isna(),
    "insurance_amount"
]

In [5]:
usd_loss_mask = (
    (df["currency_name"] == "Доллар США")
    & df["loss_payout_amt"].notna()
    & df["loss_payout_amt_usd"].isna()
)

df.loc[
    usd_loss_mask,
    "loss_payout_amt_usd"
] = df.loc[
    usd_loss_mask,
    "loss_payout_amt"
]

df["loss_payout_amt"] = df["loss_payout_amt"].fillna(0)
df["loss_payout_amt_usd"] = df["loss_payout_amt_usd"].fillna(0)

## 4. Проверка заполнения пропусков

Проверим, остались ли пропущенные значения после обработки.

In [6]:
print("Пропуски после обработки:")
print(df.isna().sum())

print(
    "\nОбщее количество пропусков:",
    df.isna().sum().sum()
)

Пропуски после обработки:
Unnamed: 0              0
contract_num            0
product_name            0
client_id               0
contract_status         0
currency_name           0
duration                0
country                 0
age                     0
sex                     0
price                   0
insurance_amount        0
loss_payout_amt         0
price_usd               0
insurance_amount_usd    0
loss_payout_amt_usd     0
dtype: int64

Общее количество пропусков: 0


In [7]:
display(
    df.loc[
        df["currency_name"] == "Доллар США",
        [
            "currency_name",
            "price",
            "price_usd",
            "insurance_amount",
            "insurance_amount_usd",
            "loss_payout_amt",
            "loss_payout_amt_usd"
        ]
    ].head(10)
)

,currency_name,price,price_usd,insurance_amount,insurance_amount_usd,loss_payout_amt,loss_payout_amt_usd
56,Доллар США,66,66.0,100000,100000.0,0.0,0.0
81,Доллар США,55,55.0,100000,100000.0,0.0,0.0
90,Доллар США,77,77.0,100000,100000.0,20000.0,20000.0
94,Доллар США,2000,2000.0,100000,100000.0,0.0,0.0
221,Доллар США,77,77.0,100000,100000.0,0.0,0.0
266,Доллар США,60,60.0,100000,100000.0,0.0,0.0
283,Доллар США,115,115.0,100000,100000.0,0.0,0.0
337,Доллар США,121,121.0,100000,100000.0,0.0,0.0
383,Доллар США,38,38.0,100000,100000.0,0.0,0.0
422,Доллар США,164,164.0,100000,100000.0,0.0,0.0


## 5. Базовая оценка качества данных

После заполнения пропусков проверим:
- наличие дубликатов;
- распределение числовых признаков;
- минимальные и максимальные значения;
- возможные очевидные аномалии.

In [8]:
print("Количество полных дубликатов:", df.duplicated().sum())

print("\nОписание числовых признаков:")
display(
    df.select_dtypes(include=np.number).describe().T
)

Количество полных дубликатов: 0

Описание числовых признаков:


,count,mean,std,min,25%,50%,75%,max
Unnamed: 0,3711.0,1.855000e+03,1.071418e+03,0.000000e+00,9.275000e+02,1.855000e+03,2.782500e+03,3.710000e+03
client_id,3711.0,5.720141e+09,4.761711e+09,1.000172e+07,1.001188e+09,1.000931e+10,1.011676e+10,1.199690e+10
duration,3711.0,2.412045e+01,6.008549e+01,7.000000e+00,1.000000e+01,1.400000e+01,1.400000e+01,3.650000e+02
age,3711.0,3.992482e+01,1.090014e+01,2.000000e+01,3.200000e+01,3.900000e+01,4.500000e+01,8.900000e+01
price,3711.0,4.167948e+03,9.555134e+03,2.700000e+01,1.096000e+03,1.918000e+03,3.836000e+03,6.000000e+04
insurance_amount,3711.0,3.296200e+06,1.511247e+06,5.000000e+04,2.000000e+06,3.000000e+06,5.000000e+06,5.000000e+06
loss_payout_amt,3711.0,2.700081e+03,4.560272e+04,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,1.500000e+06
price_usd,3711.0,5.408440e+01,1.446171e+02,6.208840e+00,1.241768e+01,2.173094e+01,4.346188e+01,2.000000e+03
insurance_amount_usd,3711.0,4.006340e+04,1.825943e+04,1.133000e+04,2.266000e+04,3.399000e+04,5.665000e+04,1.000000e+05
loss_payout_amt_usd,3711.0,4.391272e+01,7.850628e+02,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,3.000000e+04


## 6. Проверка диапазонов числовых признаков

Отдельно посмотрим минимальные и максимальные значения ключевых числовых признаков, чтобы увидеть потенциально подозрительные значения до применения алгоритма поиска аномалий.

In [9]:
numeric_cols = [
    "duration",
    "age",
    "price",
    "insurance_amount",
    "loss_payout_amt",
    "price_usd",
    "insurance_amount_usd",
    "loss_payout_amt_usd"
]

for col in numeric_cols:
    print(
        f"{col}: "
        f"min = {df[col].min()}, "
        f"max = {df[col].max()}"
    )

duration: min = 7, max = 365
age: min = 20, max = 89
price: min = 27, max = 60000
insurance_amount: min = 50000, max = 5000000
loss_payout_amt: min = 0.0, max = 1500000.0
price_usd: min = 6.208840021793029, max = 2000.0
insurance_amount_usd: min = 11330.0000397683, max = 100000.0
loss_payout_amt_usd: min = 0.0, max = 30000.0


## 7. Подготовка признаков для выявления аномалий

Для поиска аномалий используем алгоритм `IsolationForest`.

Перед обучением необходимо подготовить только информативные признаки.

Удалим:
- `Unnamed: 0` - технический индекс;
- `contract_num` - идентификатор договора;
- `client_id` - идентификатор клиента;
- `contract_status` - во всех строках статус одинаковый и не несёт информации для модели.

Категориальные признаки преобразуем в числовой формат с помощью One-Hot Encoding.

In [10]:
model_df = df.drop(
    columns=[
        "Unnamed: 0",
        "contract_num",
        "client_id",
        "contract_status"
    ]
).copy()

print("Размер данных для модели:", model_df.shape)
print("\nТипы данных:")
print(model_df.dtypes)

Размер данных для модели: (3711, 12)

Типы данных:
product_name             object
currency_name            object
duration                  int64
country                  object
age                       int64
sex                      object
price                     int64
insurance_amount          int64
loss_payout_amt         float64
price_usd               float64
insurance_amount_usd    float64
loss_payout_amt_usd     float64
dtype: object


## 8. Кодирование категориальных признаков

Алгоритм выявления аномалий работает с числовыми данными.

Преобразуем категориальные признаки через One-Hot Encoding.

In [11]:
model_encoded = pd.get_dummies(
    model_df,
    drop_first=False,
    dtype=int
)

print("Размер после кодирования:", model_encoded.shape)

display(model_encoded.head())

Размер после кодирования: (3711, 34)


,duration,age,price,insurance_amount,loss_payout_amt,price_usd,insurance_amount_usd,loss_payout_amt_usd,product_name_Страхование путешественников,product_name_Страхование путешественников USD,...,country_Канада,country_ОАЭ,country_США,country_Сирия,country_Таиланд,country_Турция,country_Узбекистан,country_Франция,sex_F,sex_M
0,10,20,1096,2000000,0.0,12.41768,22660.000080,0.0,1,0,...,0,0,0,0,0,0,0,0,0,1
1,7,55,1918,5000000,0.0,21.73094,56650.000199,0.0,1,0,...,0,0,0,0,0,0,0,0,0,1
2,10,61,1096,2000000,0.0,12.41768,22660.000080,0.0,1,0,...,0,0,0,0,0,0,0,0,1,0
3,14,25,1534,2000000,0.0,17.38022,22660.000080,0.0,1,0,...,0,0,0,0,0,0,0,0,0,1
4,7,33,1918,5000000,0.0,21.73094,56650.000199,0.0,1,0,...,0,0,0,0,0,0,0,0,0,1


In [12]:
print(
    "Остались нечисловые признаки:",
    model_encoded.select_dtypes(exclude=np.number).columns.tolist()
)

Остались нечисловые признаки: []


## 9. Выявление аномалий с помощью Isolation Forest

Для выявления аномальных наблюдений используем алгоритм `IsolationForest`.

Алгоритм пытается изолировать необычные наблюдения с помощью случайных разбиений признакового пространства. Аномальные объекты обычно изолируются быстрее обычных наблюдений.

Используем:
- `random_state=42` для воспроизводимости результата;
- `contamination="auto"`, чтобы алгоритм автоматически определил порог аномальности.

In [13]:
from sklearn.ensemble import IsolationForest

iso_forest = IsolationForest(
    contamination="auto",
    random_state=42
)

prediction = iso_forest.fit_predict(model_encoded)

prediction[:20]

array([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1])

## 10. Добавление бинарного признака anomaly

Преобразуем результат `IsolationForest` в требуемый формат:

- `1` - запись является аномальной;
- `0` - запись не является аномальной.

In [14]:
df["anomaly"] = (prediction == -1).astype(int)

print("Распределение anomaly:")
print(df["anomaly"].value_counts())

print(
    "\nДоля аномальных записей:",
    round(df["anomaly"].mean() * 100, 2),
)

Распределение anomaly:
anomaly
0    3477
1     234
Name: count, dtype: int64

Доля аномальных записей: 6.31


## 11. Анализ найденных аномалий

Посмотрим на записи, которые алгоритм классифицировал как аномальные, и сравним их ключевые числовые показатели с остальной выборкой.

In [15]:
anomalies = df[df["anomaly"] == 1]

print("Количество аномальных строк:", len(anomalies))

display(
    anomalies[
        [
            "duration",
            "age",
            "price",
            "insurance_amount",
            "loss_payout_amt",
            "price_usd",
            "insurance_amount_usd",
            "loss_payout_amt_usd",
            "anomaly"
        ]
    ].head(20)
)

Количество аномальных строк: 234


,duration,age,price,insurance_amount,loss_payout_amt,price_usd,insurance_amount_usd,loss_payout_amt_usd,anomaly
56,12,89,66,100000,0.0,66.000000,100000.000000,0.0,1
81,10,49,55,100000,0.0,55.000000,100000.000000,0.0,1
90,14,45,77,100000,20000.0,77.000000,100000.000000,20000.0,1
94,365,35,2000,100000,0.0,2000.000000,100000.000000,0.0,1
114,365,55,60000,3000000,0.0,679.800002,33990.000119,0.0,1
116,365,47,60000,3000000,0.0,679.800002,33990.000119,0.0,1
123,365,38,60000,3000000,0.0,679.800002,33990.000119,0.0,1
129,365,45,60000,3000000,0.0,679.800002,33990.000119,0.0,1
130,365,58,60000,3000000,0.0,679.800002,33990.000119,0.0,1
133,365,55,60000,3000000,0.0,679.800002,33990.000119,0.0,1


## 12. Сравнение нормальных и аномальных наблюдений

Чтобы лучше понять, какие характеристики отличают аномальные записи, сравним средние значения основных числовых признаков для нормальных и аномальных строк.

In [16]:
comparison = (
    df.groupby("anomaly")[
        [
            "duration",
            "age",
            "price",
            "insurance_amount",
            "loss_payout_amt",
            "price_usd",
            "insurance_amount_usd",
            "loss_payout_amt_usd"
        ]
    ]
    .mean()
    .T
)

comparison.columns = ["normal", "anomaly"]

display(comparison)

,normal,anomaly
duration,1.356169e+01,1.810128e+02
age,3.985217e+01,4.100427e+01
price,2.665301e+03,2.649574e+04
insurance_amount,3.409836e+06,1.607692e+06
loss_payout_amt,5.809606e+02,3.418803e+04
price_usd,3.019787e+01,4.090138e+02
insurance_amount_usd,3.863344e+04,6.131103e+04
loss_payout_amt_usd,6.582284e+00,5.986047e+02


## 13. Категориальные признаки среди аномалий

Дополнительно посмотрим распределение основных категориальных признаков среди аномальных записей.

In [17]:
for col in [
    "product_name",
    "currency_name",
    "country",
    "sex"
]:
    print(f"\n{col}:")
    print(
        df.loc[df["anomaly"] == 1, col]
        .value_counts()
        .head(10)
    )


product_name:
product_name
Страхование путешественников USD    120
Страхование путешественников        114
Name: count, dtype: int64

currency_name:
currency_name
Доллар США          120
Российский рубль    114
Name: count, dtype: int64

country:
country
США               120
Турция            102
Германия            3
Египет              2
ОАЭ                 1
Таиланд             1
Аргентина           1
Великобритания      1
Канада              1
Франция             1
Name: count, dtype: int64

sex:
sex
M    133
F    101
Name: count, dtype: int64


## 14. Интерпретация найденных аномалий

Алгоритм `IsolationForest` выявил 234 аномальные записи, что составляет 6.31% датасета.

Аномальные наблюдения в среднем отличаются от основной массы:
- большей длительностью договора;
- более высокой стоимостью полиса;
- отличающимися страховыми суммами и выплатами;
- редкими сочетаниями категориальных признаков.

При этом все 120 договоров в долларах США были отнесены алгоритмом к аномальным. Это связано с тем, что долларовые договоры представляют собой относительно редкую группу и имеют отдельное значение продукта `Страхование путешественников USD`.

Поэтому признак `anomaly = 1` следует интерпретировать как указание на статистически необычную запись, а не как однозначную ошибку в данных.

## 15. Финальная проверка данных

Перед сохранением итоговой витрины проверим:

- отсутствие пропусков;
- отсутствие полных дубликатов;
- наличие бинарного признака `anomaly`;
- количество нормальных и аномальных записей.

In [18]:
print("Размер итоговой таблицы:", df.shape)

print(
    "Общее количество пропусков:",
    df.isna().sum().sum()
)

print(
    "Полных дубликатов:",
    df.duplicated().sum()
)

print("\nРаспределение anomaly:")
print(df["anomaly"].value_counts())

print(
    "\nУникальные значения anomaly:",
    sorted(df["anomaly"].unique())
)

Размер итоговой таблицы: (3711, 17)
Общее количество пропусков: 0
Полных дубликатов: 0

Распределение anomaly:
anomaly
0    3477
1     234
Name: count, dtype: int64

Уникальные значения anomaly: [np.int64(0), np.int64(1)]


## 16. Подготовка итоговой витрины

Перед сохранением удалим технический столбец `Unnamed: 0`, который не содержит полезной информации.

Все остальные строки сохраняем, включая аномальные наблюдения, поскольку по условию задания итоговый файл должен содержать бинарный признак `anomaly`.

In [19]:
final_df = df.drop(columns=["Unnamed: 0"]).copy()

print("Размер итоговой витрины:", final_df.shape)

display(final_df.head())

Размер итоговой витрины: (3711, 16)


,contract_num,product_name,client_id,contract_status,currency_name,duration,country,age,sex,price,insurance_amount,loss_payout_amt,price_usd,insurance_amount_usd,loss_payout_amt_usd,anomaly
0,ТТЕ7227715*****,Страхование путешественников,10161870404,Действует,Российский рубль,10,Беларусь,20,M,1096,2000000,0.0,12.41768,22660.000080,0.0,0
1,ТТЕ7227715*****,Страхование путешественников,1017159879,Действует,Российский рубль,7,Индонезия,55,M,1918,5000000,0.0,21.73094,56650.000199,0.0,0
2,БАДАМСТЕ55*****,Страхование путешественников,10161883357,Действует,Российский рубль,10,Беларусь,61,F,1096,2000000,0.0,12.41768,22660.000080,0.0,0
3,ТТЕ7227715*****,Страхование путешественников,100766461,Действует,Российский рубль,14,Грузия,25,M,1534,2000000,0.0,17.38022,22660.000080,0.0,0
4,ТТЕ7227715*****,Страхование путешественников,10171604758,Действует,Российский рубль,7,Аргентина,33,M,1918,5000000,0.0,21.73094,56650.000199,0.0,0


## 17. Сохранение результата

Сохраним очищенную витрину в Excel-файл.

Файл содержит:
- заполненные пропущенные значения;
- исходные признаки;
- бинарный признак `anomaly`, где:
  - `1` - аномальная запись;
  - `0` - нормальная запись.

In [20]:
output_file = "cleaned_insurance_data.xlsx"

final_df.to_excel(
    output_file,
    index=False
)

print("Файл сохранен:", output_file)

Файл сохранен: cleaned_insurance_data.xlsx


## 18. Итог

В рамках практической работы была выполнена очистка витрины страховых полисов.

В ходе работы:

- проанализированы пропущенные значения и причины их появления;
- восстановлены долларовые показатели для договоров в долларах США;
- отсутствующие страховые выплаты заполнены нулевыми значениями;
- после обработки в таблице не осталось пропусков;
- полные дубликаты не обнаружены;
- удалены несущественные признаки перед построением модели аномалий;
- категориальные признаки преобразованы с помощью One-Hot Encoding;
- для выявления аномалий применён алгоритм `IsolationForest`;
- выявлено 234 аномальных наблюдения, что составляет 6.31% датасета;
- добавлен бинарный признак `anomaly`;
- технический столбец `Unnamed: 0` удалён из итоговой витрины;
- результат сохранён в файл `cleaned_insurance_data.xlsx`.

Важно учитывать, что `anomaly = 1` означает статистически необычную запись, а не обязательно ошибку в данных.